# 1.11 Uniform convergence

The engine under every bound: training error converges to true error simultaneously for all hypotheses in the class.

Hoeffding handles one fixed hypothesis; the union bound extends the guarantee to every hypothesis at once. This is why ERM can safely choose after seeing the sample when the class is not too large for the sample size.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(16)


def theory_ladder(topic):
    """Return compact D1--D5 inputs for F16 theory notebooks."""
    if topic == "srm":
        return [
            {"name": "D1 lesson four-rung ladder", "emp": np.array([0.30, 0.15, 0.07, 0.05]), "H": np.array([2, 16, 256, 65536]), "m": 200, "delta": 0.05},
            {"name": "D2 six nested polynomial rungs", "emp": np.array([0.34, 0.21, 0.13, 0.09, 0.075, 0.07]), "H": np.array([2, 8, 32, 128, 512, 2048]), "m": 200, "delta": 0.05},
            {"name": "D3 larger sample shifts upward", "emp": np.array([0.30, 0.15, 0.07, 0.04, 0.035]), "H": np.array([2, 16, 256, 65536, 1048576]), "m": 2000, "delta": 0.05},
            {"name": "D4 noisy empirical errors", "emp": np.array([0.31, 0.19, 0.12, 0.115, 0.13]), "H": np.array([2, 16, 256, 4096, 65536]), "m": 300, "delta": 0.05},
            {"name": "D5 bad ordering stress case", "emp": np.array([0.28, 0.10, 0.03, 0.02, 0.00]), "H": np.array([4, 4096, 64, 1048576, 256]), "m": 120, "delta": 0.05},
        ]
    if topic == "regularization":
        x = np.linspace(-1.0, 1.0, 25)
        base = 1.0 + 2.0 * x - 1.5 * x ** 2
        return [
            {"name": "D1 lesson 3-row regression", "X": np.array([[1.0, 1.0], [1.0, 2.0], [1.0, 3.0]]), "y": np.array([1.0, 2.0, 2.0]), "lambdas": np.array([0.0, 1.0, 10.0, 100.0])},
            {"name": "D2 dense lambda path", "X": np.column_stack([np.ones_like(x), x]), "y": 1.0 + 2.0 * x + 0.15 * np.sin(7.0 * x), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D3 higher polynomial capacity", "X": np.column_stack([np.ones_like(x), x, x ** 2, x ** 3, x ** 4, x ** 5]), "y": base + 0.10 * np.sin(9.0 * x), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D4 noisy labels", "X": np.column_stack([np.ones_like(x), x, x ** 2, x ** 3]), "y": base + rng.normal(0.0, 0.18, size=x.shape), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D5 unscaled-feature trap", "X": np.column_stack([np.ones_like(x), x, 100.0 * x ** 2]), "y": base + rng.normal(0.0, 0.10, size=x.shape), "lambdas": np.logspace(-3, 2, 20)},
        ]
    if topic == "stability":
        sample = np.array([2.0, 4.0, 6.0, 8.0, 10.0])
        long_sample = np.linspace(0.0, 1.0, 50)
        return [
            {"name": "D1 lesson 5-number mean", "sample": sample, "kind": "mean"},
            {"name": "D2 m=50 bounded-range mean", "sample": long_sample, "kind": "mean"},
            {"name": "D3 ridge lambda ladder", "sample": np.column_stack([np.ones(20), np.linspace(-1.0, 1.0, 20)]), "target": 1.0 + np.linspace(-1.0, 1.0, 20), "lambdas": np.array([0.1, 0.3, 1.0, 3.0]), "kind": "ridge"},
            {"name": "D4 compare mean/ridge to 1-NN-style rule", "sample": np.linspace(0.0, 1.0, 20), "kind": "nn_compare"},
            {"name": "D5 outlier removal stress case", "sample": np.array([0.0, 0.1, 0.2, 0.3, 9.0]), "kind": "outlier"},
        ]
    if topic == "nfl":
        return [
            {"name": "D1 one unseen point", "n": 1, "prediction": np.array([0])},
            {"name": "D2 lesson three unseen points", "n": 3, "prediction": np.array([0, 0, 0])},
            {"name": "D3 four unseen points", "n": 4, "prediction": np.array([1, 0, 1, 0])},
            {"name": "D4 biased smooth subset", "n": 4, "prediction": np.array([0, 0, 1, 1]), "subset": "smooth"},
            {"name": "D5 adversarial mirror subset", "n": 4, "prediction": np.array([0, 0, 1, 1]), "subset": "mirror"},
        ]
    if topic == "uniform":
        return [
            {"name": "D1 one fixed hypothesis", "H": 1, "m": 500, "epsilon": 0.1},
            {"name": "D2 hundred-hypothesis lesson class", "H": 100, "m": 500, "epsilon": 0.1},
            {"name": "D3 low-m vacuous rungs", "H": 100, "ms": np.array([100, 150, 230, 300]), "epsilon": 0.1},
            {"name": "D4 solve m for delta", "H": 100, "delta": 0.05, "epsilon": 0.1},
            {"name": "D5 correlated non-iid violation", "H": 100, "m": 500, "epsilon": 0.1, "rho": 0.9},
        ]
    raise ValueError(topic)


def all_binary_targets(n):
    rows = list(itertools.product([0, 1], repeat=n))
    return np.array(rows, dtype=int)


def ridge_weights(X, y, lam, penalize_intercept=True):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    penalty = np.eye(X.shape[1])
    if not penalize_intercept:
        penalty[0, 0] = 0.0
    system = X.T @ X + lam * penalty
    rhs = X.T @ y
    weights = np.linalg.solve(system, rhs)
    return weights


def print_rows(rows, headers):
    widths = [len(h) for h in headers]
    for row in rows:
        for i, item in enumerate(row):
            widths[i] = max(widths[i], len(str(item)))
    header = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
    print(header)
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(" | ".join(str(item).ljust(widths[i]) for i, item in enumerate(row)))


## The concept, built once (D1)

For bounded loss, Hoeffding gives $2e^{-2m\varepsilon^2}$ for one hypothesis. Uniform convergence over a finite class uses
$$\Pr\left(\sup_{h\in H}|R_S(h)-R(h)|>\varepsilon\right)\le |H|\cdot2e^{-2m\varepsilon^2}.$$
The lesson check is $|H|=100$, $m=500$, $\varepsilon=0.1$, which yields $0.0091$.

In [ ]:

def uniform_failure(H_size, m, epsilon):
    value = H_size * 2.0 * math.exp(-2.0 * m * epsilon ** 2)
    return float(value)


def required_m_for_uniform(H_size, epsilon, delta):
    numerator = math.log(H_size) + math.log(2.0 / delta)
    denominator = 2.0 * epsilon ** 2
    return float(numerator / denominator)


def solved_epsilon(H_size, m, delta):
    numerator = math.log(H_size) + math.log(2.0 / delta)
    denominator = 2.0 * m
    return float(math.sqrt(numerator / denominator))


Now assert the lesson's exact scale: one hypothesis at $m=500$ has $2e^{-10}=0.00009$ and $100$ hypotheses have failure bound $0.0091$; solving for $\delta=0.05$ gives $414.7$, so $415$ samples.

In [ ]:

one_h = uniform_failure(1, 500, 0.1)
hundred_h = uniform_failure(100, 500, 0.1)
needed_m = required_m_for_uniform(100, 0.1, 0.05)
print(f"One hypothesis failure: {one_h:.5f}")
print(f"Hundred-hypothesis uniform failure: {hundred_h:.4f}")
print(f"Required m for delta=0.05: {needed_m:.1f}, ceiling {math.ceil(needed_m)}")
assert np.isclose(round(one_h, 5), 0.00009)
assert np.isclose(round(hundred_h, 4), 0.0091)
assert np.isclose(round(needed_m, 1), 414.7)
assert math.ceil(needed_m) == 415


## The dataset ladder

D1 starts with one fixed hypothesis. D2 is the lesson class. D3 shows low-sample vacuity, D4 solves required sample size, and D5 illustrates a correlated non-i.i.d. violation.

In [ ]:

uniform_ladder = theory_ladder("uniform")
rows = []
for dataset in uniform_ladder:
    if "ms" in dataset:
        size_info = dataset["ms"].tolist()
    else:
        size_info = dataset.get("m", "solve")
    rows.append((dataset["name"], dataset["H"], size_info, dataset["epsilon"]))
print_rows(rows, ["dataset", "|H|", "m info", "epsilon"])


In [ ]:

uniform_results = []
for dataset in uniform_ladder:
    if "ms" in dataset:
        metrics = np.array([uniform_failure(dataset["H"], int(m), dataset["epsilon"]) for m in dataset["ms"]])
        metric = float(metrics[-1])
        detail = metrics
    elif "delta" in dataset:
        metric = required_m_for_uniform(dataset["H"], dataset["epsilon"], dataset["delta"])
        detail = metric
    elif "rho" in dataset:
        effective_m = dataset["m"] * (1.0 - dataset["rho"]) / (1.0 + dataset["rho"])
        metric = uniform_failure(dataset["H"], effective_m, dataset["epsilon"])
        detail = effective_m
    else:
        metric = uniform_failure(dataset["H"], dataset["m"], dataset["epsilon"])
        detail = metric
    uniform_results.append({"dataset": dataset, "metric": metric, "detail": detail})
    print(f"{dataset['name']}: metric {metric:.4f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for j, result in enumerate(uniform_results):
    dataset = result["dataset"]
    if "ms" in dataset:
        axes[0, j].bar(dataset["ms"], result["detail"])
        axes[0, j].axhline(1.0, color="red", linestyle="--")
    elif "delta" in dataset:
        axes[0, j].bar(["needed"], [result["metric"]])
    else:
        per_h = uniform_failure(1, dataset.get("m", 500), dataset["epsilon"])
        axes[0, j].bar(["one h", "union"], [per_h, min(result["metric"], 1.5)])
    axes[0, j].set_title(dataset["name"].split()[0])
    axes[0, j].set_ylabel("failure bound")
ms = np.arange(100, 801, 50)
curve = np.array([uniform_failure(100, int(m), 0.1) for m in ms])
axes[1, 0].plot(ms, curve, marker="o")
axes[1, 0].axhline(0.05, color="red", linestyle="--", label="delta=0.05")
axes[1, 0].set_xlabel("m")
axes[1, 0].set_ylabel("uniform failure probability")
axes[1, 0].legend()
for ax in axes[1, 1:]:
    ax.axis("off")
fig.suptitle("Uniform-convergence panels and failure-probability-vs-m curve")
plt.tight_layout()


## Pitfall on D5: dropping the i.i.d. assumption or accepting vacuity

Hoeffding needs independent bounded losses. Correlation lowers the effective sample size, making a bound that looked strong become weak or vacuous. The fix is to restore i.i.d. sampling, block correlated data, or switch to a bound designed for dependence.

In [ ]:

plain = uniform_failure(100, 500, 0.1)
rho = 0.9
effective_m = 500 * (1.0 - rho) / (1.0 + rho)
correlated = uniform_failure(100, effective_m, 0.1)
vc_growth_bound = 2.0 * sum(math.comb(500, i) for i in range(0, 4))
vc_failure = vc_growth_bound * math.exp(-2.0 * 500 * 0.1 ** 2)
print(f"I.i.d. union-bound failure: {plain:.4f}")
print(f"Correlated effective m: {effective_m:.1f}")
print(f"Correlated illustrative failure: {correlated:.3f}")
print(f"VC-style growth substitution for d=3 remains finite: {vc_failure:.3f}")


## Evaluate it + practice

- Metric: uniform-convergence failure probability or solved epsilon penalty.
- No-skill baseline: one fixed hypothesis uses Hoeffding without the |H| multiplier.
- Cheap sanity check: failure probability decreases with m and increases with |H|.
- Ablation: drop the union-bound factor and see a falsely optimistic guarantee.
- Failure signals: right side is at least one, losses are unbounded, or samples are correlated.

### Practice prompts

- Solve the required m for epsilon 0.05 and compare with epsilon 0.1.

- Find the m where the D3 curve first drops below 0.25.

- Replace |H| with a growth-function value and compare the bound.
